# Лабораторная работа №1
Самостоятельно написать код, реализующий искусственный нейрон с сигма-функцией активации, и возможность строить на его основе многослойные сети. Код должен также реализовывать градиентный спуск и обратное распространение ошибки.
На основе вашего кода:
1.	Решить задачу  классификации датаcета Iris одним нейроном.
2.	Решить задачу  классификации датаcета Iris одним  нейросетью из 2 слоев по 10 нейронов в слое.
3.	Отрисовать разделяющую линию для обеих моделей. Сравнить метрики классификации.


In [1]:
from sklearn.datasets import load_iris
iris = load_iris()

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from typing import Self, Any
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

### Single neuron

In [2]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def sigmoid_derivative(a):
    return a * (1.0 - a)


class Neuron:
    def __init__(self, n_features, lr=0.1, n_epochs=1000) -> None:
        rng = np.random.default_rng(42)
        self.w = rng.normal(0, 0.01, n_features)
        self.b = 0.0
        self.lr = lr
        self.n_epochs = n_epochs
        self.loss_history = []

    def predict_proba(self, X) -> Any:
        return sigmoid(X @ self.w + self.b)

    def predict(self, X, threshold=0.5) -> Any:
        return (self.predict_proba(X) >= threshold).astype(int)

    def fit(self, X, y) -> Self:
        m = len(y)
        for _ in range(self.n_epochs):
            a = self.predict_proba(X) # forward
            loss = -np.mean(y * np.log(a + 1e-9) + (1 - y) * np.log(1 - a + 1e-9))
            self.loss_history.append(loss)
            delta = (a - y) / m
            self.w -= self.lr * (X.T @ delta)
            self.b -= self.lr * delta.sum()
        return self

### NN

In [3]:
class NeuralNetwork:
    def __init__(self, layer_sizes, lr=0.05, n_epochs=2000) -> None:
        self.layer_sizes = layer_sizes
        self.lr = lr
        self.n_epochs = n_epochs
        self.loss_history = []
        self._init_weights()

    def _init_weights(self) -> None:
        rng = np.random.default_rng(42)
        self.weights = []
        self.biases = []
        for i in range(len(self.layer_sizes) - 1):
            fan_in  = self.layer_sizes[i]
            fan_out = self.layer_sizes[i + 1]
            scale = np.sqrt(2.0 / (fan_in + fan_out))
            self.weights.append(rng.normal(0, scale, (fan_in, fan_out)))
            self.biases.append(np.zeros(fan_out))

    def _forward(self, X) -> list:
        activations = [X]
        a = X
        for W, b in zip(self.weights, self.biases):
            z = a @ W + b
            a = sigmoid(z)
            activations.append(a)
        return activations  # activations[-1] - выход сети

    def _backward(self, activations, y) -> tuple[list[None], list[None]]:
        m = len(y)
        grads_w = [None] * len(self.weights)
        grads_b = [None] * len(self.biases)

        # Ошибка выходного слоя
        delta = (activations[-1] - y.reshape(-1, 1)) / m  # (m, 1)

        for i in reversed(range(len(self.weights))):
            grads_w[i] = activations[i].T @ delta # (fan_in, fan_out)
            grads_b[i] = delta.sum(axis=0)
            if i > 0:
                delta = (delta @ self.weights[i].T) * sigmoid_derivative(activations[i])
        return grads_w, grads_b

    def fit(self, X, y) -> Self:
        for _ in range(self.n_epochs):
            acts = self._forward(X)
            a_out = acts[-1].ravel()
            loss = -np.mean(y * np.log(a_out + 1e-9) +
                            (1 - y) * np.log(1 - a_out + 1e-9))
            self.loss_history.append(loss)
            gw, gb = self._backward(acts, y)
            for i in range(len(self.weights)):
                self.weights[i] -= self.lr * gw[i]
                self.biases[i]  -= self.lr * gb[i]
        return self

    def predict_proba(self, X) -> Any:
        return self._forward(X)[-1].ravel()

    def predict(self, X, threshold=0.5) -> Any:
        return (self.predict_proba(X) >= threshold).astype(int)

### Running

In [4]:
iris = load_iris()
X_all  = iris.data
y_bin  = (iris.target != 0).astype(float)

# Используем только 2 признака для отрисовки разделяющей линии
FEAT_X, FEAT_Y = 0, 1 # sepal length и sepal width
X2 = X_all[:, [FEAT_X, FEAT_Y]]
X4 = X_all

scaler2 = StandardScaler()
scaler4 = StandardScaler()
X2_s = scaler2.fit_transform(X2)
X4_s = scaler4.fit_transform(X4)

X2_tr, X2_te, y_tr, y_te = train_test_split(X2_s, y_bin, test_size=0.25, random_state=42)
X4_tr, X4_te, _, _ = train_test_split(X4_s, y_bin, test_size=0.25, random_state=42)


# Обучение
neuron = Neuron(n_features=4, lr=0.5, n_epochs=2000)
neuron.fit(X4_tr, y_tr)
y_pred_n = neuron.predict(X4_te)
print("[Модель 1] Одиночный нейрон (4 признака)")
print(classification_report(y_te, y_pred_n,
        target_names=["setosa", "non-setosa"]))

nn = NeuralNetwork([4, 10, 10, 1], lr=0.1, n_epochs=3000)
nn.fit(X4_tr, y_tr)
y_pred_nn = nn.predict(X4_te)
print("[Модель 2] Нейросеть 4→10→10→1 (2 слоя по 10 нейронов)")
print(classification_report(y_te, y_pred_nn,
        target_names=["setosa", "non-setosa"]))

print(f"{'Метрика':<20} {'Нейрон':>15} {'Нейросеть 2 слоя':>18}")
for metric_name, fn in [("Accuracy",  accuracy_score),
                            ("Precision", lambda a, b: precision_score(a, b, zero_division=0)),
                            ("Recall",    lambda a, b: recall_score(a, b, zero_division=0)),
                            ("F1-score",  lambda a, b: f1_score(a, b, zero_division=0))]:
    v1 = fn(y_te, y_pred_n)
    v2 = fn(y_te, y_pred_nn)
    print(f"{metric_name:<20} {v1:>15.4f} {v2:>18.4f}")


# 6.  Визуализация: разделяющие линии (2 признака)
neuron2 = Neuron(n_features=2, lr=0.5, n_epochs=2000)
neuron2.fit(X2_tr, y_tr)
nn2 = NeuralNetwork([2, 10, 10, 1], lr=0.1, n_epochs=3000)
nn2.fit(X2_tr, y_tr)

h = 0.05
x_min, x_max = X2_s[:, 0].min() - 0.5, X2_s[:, 0].max() + 0.5
y_min, y_max = X2_s[:, 1].min() - 0.5, X2_s[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                        np.arange(y_min, y_max, h))
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Лабораторная работа №1 — Ирисы (sepal length vs sepal width)", fontsize=14, fontweight="bold")

colors = ["#FF8888", "#8888FF"]
cmap_bg = plt.cm.RdBu

# Singular neuron
Z1 = neuron2.predict_proba(grid).reshape(xx.shape)
ax = axes[0]
ax.contourf(xx, yy, Z1, levels=[0, 0.5, 1], colors=colors, alpha=0.4)
ax.contour( xx, yy, Z1, levels=[0.5], colors="k", linewidths=2)
for cls, label, c in [(0, "Setosa", "red"), (1, "Non-setosa", "blue")]:
    mask = y_bin == cls
    ax.scatter(X2_s[mask, 0], X2_s[mask, 1], c=c,
                edgecolors="k", s=40, label=label)
acc1 = accuracy_score(y_te, neuron2.predict(X2_te))
ax.set_title(f"Одиночный нейрон\nAcc (2 призн.) = {acc1:.3f}")
ax.set_xlabel(iris.feature_names[FEAT_X])
ax.set_ylabel(iris.feature_names[FEAT_Y])
ax.legend(fontsize=8)

# NN
Z2 = nn2.predict_proba(grid).reshape(xx.shape)
ax = axes[1]
ax.contourf(xx, yy, Z2, levels=[0, 0.5, 1], colors=colors, alpha=0.4)
ax.contour( xx, yy, Z2, levels=[0.5], colors="k", linewidths=2)
for cls, label, c in [(0, "Setosa", "red"), (1, "Non-setosa", "blue")]:
    mask = y_bin == cls
    ax.scatter(X2_s[mask, 0], X2_s[mask, 1], c=c,
                edgecolors="k", s=40, label=label)
acc2 = accuracy_score(y_te, nn2.predict(X2_te))
ax.set_title(f"Нейросеть 2→10→10→1\nAcc (2 призн.) = {acc2:.3f}")
ax.set_xlabel(iris.feature_names[FEAT_X])
ax.legend(fontsize=8)

# Loss
ax = axes[2]
ax.plot(neuron.loss_history, label="Нейрон (4 призн.)", color="orangered")
ax.plot(nn.loss_history, label="Нейросеть (4 призн.)", color="steelblue")
ax.set_title("Кривые потерь (бинарная кросс-энтропия)")
ax.set_xlabel("Эпоха")
ax.set_ylabel("Loss")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("lab1_iris.png", dpi=150, bbox_inches="tight")
plt.close()
print()
print("Графики сохранены: lab1_iris.png")

[Модель 1] Одиночный нейрон (4 признака)
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        15
  non-setosa       1.00      1.00      1.00        23

    accuracy                           1.00        38
   macro avg       1.00      1.00      1.00        38
weighted avg       1.00      1.00      1.00        38

[Модель 2] Нейросеть 4→10→10→1 (2 слоя по 10 нейронов)
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        15
  non-setosa       1.00      1.00      1.00        23

    accuracy                           1.00        38
   macro avg       1.00      1.00      1.00        38
weighted avg       1.00      1.00      1.00        38

Метрика                       Нейрон   Нейросеть 2 слоя
Accuracy                      1.0000             1.0000
Precision                     1.0000             1.0000
Recall                        1.0000             1.0000
F1-score                  